# CZ Benchmarks GenePT 1024d Embedding Generation

This notebook implements Phase 1 of the CZ benchmarks evaluation: generating GenePT embeddings at 1024 dimensions by truncating existing 3072d embeddings (Matryoshka property) for all 5 tissue types.

Based on `specs/cz_benchmarks_evaluation_spec.md`

In [15]:
# Setup notebook environment
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

# Add parent directory to path
repo_dir = Path.cwd().parent
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print(f"Repository directory: {repo_dir}")
data_dir = repo_dir / "data"
print(f"Data directory: {data_dir}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Repository directory: /Users/rj/personal/GenePT-tools
Data directory: /Users/rj/personal/GenePT-tools/data


In [16]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from src.inference import create_embedding_matrix, create_cell_embeddings
from tqdm.auto import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration
TISSUE_TYPES = ['Blood', 'Bone_Marrow', 'Lung', 'Mammary', 'Thymus']
BENCHMARK_DIR = Path("/Users/rj/personal/Tabula_Sapiens_v2_Curated_Benchmark")
OUTPUT_DIR = data_dir / 'cz_benchmark' / 'embeddings' / 'genept_1024d'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Benchmark data directory: {BENCHMARK_DIR}")
print(f"Will save GenePT 1024d embeddings to: {OUTPUT_DIR}")

Benchmark data directory: /Users/rj/personal/Tabula_Sapiens_v2_Curated_Benchmark
Will save GenePT 1024d embeddings to: /Users/rj/personal/GenePT-tools/data/cz_benchmark/embeddings/genept_1024d


## Load and Prepare GenePT Embeddings

## Dataset Loading Function

In [17]:
def load_tabula_sapiens_dataset(tissue_name: str) -> ad.AnnData:
    """Load a Tabula Sapiens v2 curated benchmark dataset."""
    # Find the file for this tissue
    pattern = f"*_{tissue_name}_v2_curated.h5ad"
    files = list(BENCHMARK_DIR.glob(pattern))
    
    if not files:
        raise FileNotFoundError(f"No file found for tissue: {tissue_name}")
    
    file_path = files[0]
    print(f"Loading {tissue_name} from: {file_path.name}")
    
    # Load the dataset
    adata = sc.read_h5ad(file_path)
    
    # Check which layer contains the expression data
    if adata.X is None or (hasattr(adata.X, 'nnz') and adata.X.nnz == 0):
        # Common layer names for expression data - X_original is used in Tabula Sapiens v2
        expression_layers = ['X_original', 'counts', 'raw_counts', 'data']
        
        for layer_name in expression_layers:
            if layer_name in adata.layers:
                print(f"Using layer '{layer_name}' as expression matrix")
                adata.X = adata.layers[layer_name].copy()
                break
    
    # Check if cell_type column exists, if not look for alternatives
    if 'cell_type' not in adata.obs.columns:
        for col in ['cell_ontology_class', 'celltype', 'CellType', 'free_annotation']:
            if col in adata.obs.columns:
                adata.obs['cell_type'] = adata.obs[col]
                print(f"Using column '{col}' as cell_type")
                break
    
    print(f"Loaded {tissue_name}: {adata.n_obs} cells, {adata.n_vars} genes")
    print(f"Cell types: {adata.obs['cell_type'].nunique()}")
    
    return adata

In [18]:
def load_genept_3072d_embeddings():
    """Load existing 3072d GenePT gene embeddings."""
    embeddings_path = data_dir / 'huggingface_model' / 'embedding_associations_cell_type_tissue_drug_pathway_openai_large.parquet'
    
    print(f"Loading GenePT embeddings from: {embeddings_path}")
    gene_embeddings_df = pd.read_parquet(embeddings_path)
    print(f"Loaded embeddings shape: {gene_embeddings_df.shape}")
    print(f"Index name: {gene_embeddings_df.index.name}")
    print(f"Number of genes: {len(gene_embeddings_df)}")
    print(f"Example genes: {gene_embeddings_df.index[:5].tolist()}")
    
    return gene_embeddings_df

def truncate_to_1024d(gene_embeddings_df):
    """Truncate 3072d embeddings to first 1024 dimensions using Matryoshka property."""
    # Get embedding columns (0-3071) and truncate to first 1024 dimensions
    embedding_cols = [col for col in gene_embeddings_df.columns if str(col).isdigit() and int(col) >= 0]
    embedding_cols_1024 = [col for col in embedding_cols if int(col) < 1024]  # First 1024 dimensions
    
    print(f"Original embedding dimensions: {len(embedding_cols)}")
    print(f"Truncated to dimensions: {len(embedding_cols_1024)}")
    
    # Create truncated embedding dataframe (gene names are already in index)
    gene_embeddings_1024d = gene_embeddings_df[embedding_cols_1024].copy()
    print(f"Truncated embeddings shape: {gene_embeddings_1024d.shape}")
    print(f"Gene names preserved in index: {gene_embeddings_1024d.index.name}")
    
    return gene_embeddings_1024d

# Load and truncate GenePT embeddings
gene_embeddings_3072d = load_genept_3072d_embeddings()
gene_embeddings_1024d = truncate_to_1024d(gene_embeddings_3072d)

Loading GenePT embeddings from: /Users/rj/personal/GenePT-tools/data/huggingface_model/embedding_associations_cell_type_tissue_drug_pathway_openai_large.parquet
Loaded embeddings shape: (33703, 3072)
Index name: gene_name
Number of genes: 33703
Example genes: ['A1BG', 'A1BG-AS1', 'A1CF', 'A2M', 'A2M-AS1']
Original embedding dimensions: 3072
Truncated to dimensions: 1024
Truncated embeddings shape: (33703, 1024)
Gene names preserved in index: gene_name


## Gene Name Processing Function

In [19]:
def process_gene_names(adata):
    """Process gene names to handle mixed formats in feature_name column.
    
    Following the pattern from cz_benchmark_evaluation.ipynb to ensure consistency.
    """
    gene_list = []
    for gene in adata.var['feature_name']:
        # Handle different formats:
        # 1. "GENE_ENSG00000123456" -> extract "GENE" 
        # 2. "ENSG00000123456.15" -> use as-is (will likely not match)
        # 3. "GENE" -> use as-is
        if '_ENSG' in gene:
            # Extract gene symbol before _ENSG
            gene_symbol = gene.split('_ENSG')[0]
            gene_list.append(gene_symbol)
        else:
            # Use as-is for normal gene symbols or ENSG IDs
            gene_list.append(gene)
    
    print(f"Processed feature_name column for gene symbols")
    print(f"Example genes after processing: {gene_list[:5]}")
    print(f"Total genes processed: {len(gene_list)}")
    
    return gene_list

## Generate 1024d Cell Embeddings for Each Tissue

In [20]:
def generate_genept_1024d_for_tissue(tissue_type, gene_embeddings_1024d):
    """Generate GenePT 1024d cell embeddings for a specific tissue type."""
    print(f"\n=== Processing {tissue_type} ===")
    
    # Load Tabula Sapiens dataset
    adata = load_tabula_sapiens_dataset(tissue_type)
    
    # Process gene names
    gene_list = process_gene_names(adata)
    
    # Create embedding matrix
    # Gene names are in the index, so reset it to make it a column for create_embedding_matrix
    gene_embeddings_with_names = gene_embeddings_1024d.reset_index()
    
    print(f"Creating embedding matrix...")
    embedding_matrix, valid_indices = create_embedding_matrix(
        gene_embeddings_with_names, gene_list, id_column='gene_name'
    )
    
    print(f"Valid genes matched: {len(valid_indices)} out of {len(gene_list)} ({len(valid_indices)/len(gene_list)*100:.1f}%)")
    print(f"Embedding matrix shape: {embedding_matrix.shape}")
    
    # Check if we have sufficient gene coverage
    if len(valid_indices) == 0:
        raise ValueError(f"No genes matched between {tissue_type} dataset and embeddings!")
    
    if len(valid_indices) < 100:
        print(f"WARNING: Low gene coverage for {tissue_type}: {len(valid_indices)} genes")
    
    # Create cell embeddings
    print(f"Creating cell embeddings...")
    cell_embeddings = create_cell_embeddings(
        adata.X, embedding_matrix, valid_indices
    )
    
    print(f"Cell embeddings shape: {cell_embeddings.shape}")
    
    # Create output dataframe with cell barcodes and metadata
    embedding_df = pd.DataFrame(
        cell_embeddings,
        index=adata.obs_names,
        columns=[f'genept_{i}' for i in range(1024)]
    )
    
    # Add essential metadata
    embedding_df['cell_type'] = adata.obs['cell_type'].values
    embedding_df['tissue_type'] = tissue_type
    
    # Add donor information if available
    if 'donor_id' in adata.obs.columns:
        embedding_df['donor_id'] = adata.obs['donor_id'].values
    
    return embedding_df

# Generate embeddings for all tissues
tissue_embeddings = {}

for tissue_type in tqdm(TISSUE_TYPES, desc="Processing tissues"):
    try:
        embedding_df = generate_genept_1024d_for_tissue(tissue_type, gene_embeddings_1024d)
        tissue_embeddings[tissue_type] = embedding_df
        
        # Save immediately
        output_path = OUTPUT_DIR / f'genept_1024d_{tissue_type}_embeddings.parquet'
        embedding_df.to_parquet(output_path)
        print(f"Saved {tissue_type} embeddings to: {output_path}")
        
    except Exception as e:
        print(f"ERROR processing {tissue_type}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n=== Generation Complete ===")
print(f"Successfully processed {len(tissue_embeddings)} out of {len(TISSUE_TYPES)} tissues")

Processing tissues:   0%|          | 0/5 [00:00<?, ?it/s]


=== Processing Blood ===
Loading Blood from: homo_sapiens_10df7690-6d10-4029-a47e-0f071bb2df83_Blood_v2_curated.h5ad
Loaded Blood: 17802 cells, 27284 genes
Cell types: 19
Processed feature_name column for gene symbols
Example genes after processing: ['DPM1', 'SCYL3', 'C1orf112', 'FGR', 'CFH']
Total genes processed: 27284
Creating embedding matrix...
Creating embedding matrix for 27284 genes
Selecting embedding inidices
Embedding matrix shape: (1024, 18402)
Valid genes matched: 18402 out of 27284 (67.4%)
Embedding matrix shape: (1024, 18402)
Creating cell embeddings...
Cell embeddings shape: (17802, 1024)


Processing tissues:  20%|██        | 1/5 [00:20<01:22, 20.53s/it]

Saved Blood embeddings to: /Users/rj/personal/GenePT-tools/data/cz_benchmark/embeddings/genept_1024d/genept_1024d_Blood_embeddings.parquet

=== Processing Bone_Marrow ===
Loading Bone_Marrow from: homo_sapiens_10df7690-6d10-4029-a47e-0f071bb2df83_Bone_Marrow_v2_curated.h5ad
Loaded Bone_Marrow: 8045 cells, 26167 genes
Cell types: 25
Processed feature_name column for gene symbols
Example genes after processing: ['TSPAN6', 'DPM1', 'SCYL3', 'C1orf112', 'FGR']
Total genes processed: 26167
Creating embedding matrix...
Creating embedding matrix for 26167 genes
Selecting embedding inidices
Embedding matrix shape: (1024, 18086)
Valid genes matched: 18086 out of 26167 (69.1%)
Embedding matrix shape: (1024, 18086)
Creating cell embeddings...
Cell embeddings shape: (8045, 1024)


Processing tissues:  40%|████      | 2/5 [00:30<00:42, 14.15s/it]

...

Saved Lung embeddings to: /Users/rj/personal/GenePT-tools/data/cz_benchmark/embeddings/genept_1024d/genept_1024d_Lung_embeddings.parquet

=== Processing Mammary ===
Loading Mammary from: homo_sapiens_10df7690-6d10-4029-a47e-0f071bb2df83_Mammary_v2_curated.h5ad
Loaded Mammary: 18539 cells, 28389 genes
Cell types: 15
Processed feature_name column for gene symbols
Example genes after processing: ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112']
Total genes processed: 28389
Creating embedding matrix...
Creating embedding matrix for 28389 genes
Selecting embedding inidices
Embedding matrix shape: (1024, 19271)
Valid genes matched: 19271 out of 28389 (67.9%)
Embedding matrix shape: (1024, 19271)
Creating cell embeddings...
Cell embeddings shape: (18539, 1024)


Processing tissues:  80%|████████  | 4/5 [01:17<00:21, 21.01s/it]

Saved Mammary embeddings to: /Users/rj/personal/GenePT-tools/data/cz_benchmark/embeddings/genept_1024d/genept_1024d_Mammary_embeddings.parquet

=== Processing Thymus ===
Loading Thymus from: homo_sapiens_10df7690-6d10-4029-a47e-0f071bb2df83_Thymus_v2_curated.h5ad
Loaded Thymus: 9933 cells, 30453 genes
Cell types: 27
Processed feature_name column for gene symbols
Example genes after processing: ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112']
Total genes processed: 30453
Creating embedding matrix...
Creating embedding matrix for 30453 genes
Selecting embedding inidices
Embedding matrix shape: (1024, 19981)
Valid genes matched: 19981 out of 30453 (65.6%)
Embedding matrix shape: (1024, 19981)
Creating cell embeddings...
Cell embeddings shape: (9933, 1024)


Processing tissues: 100%|██████████| 5/5 [01:31<00:00, 18.38s/it]

Saved Thymus embeddings to: /Users/rj/personal/GenePT-tools/data/cz_benchmark/embeddings/genept_1024d/genept_1024d_Thymus_embeddings.parquet

=== Generation Complete ===
Successfully processed 5 out of 5 tissues


## Validation and Summary

In [21]:
# Validate generated embeddings
summary_data = []

for tissue_type, embedding_df in tissue_embeddings.items():
    summary_data.append({
        'Tissue': tissue_type,
        'Cells': len(embedding_df),
        'Cell_Types': embedding_df['cell_type'].nunique(),
        'Embedding_Dims': len([col for col in embedding_df.columns if col.startswith('genept_')]),
        'Has_Donor_Info': 'donor_id' in embedding_df.columns,
        'File_Size_MB': round((OUTPUT_DIR / f'genept_1024d_{tissue_type}_embeddings.parquet').stat().st_size / 1024 / 1024, 1)
    })

summary_df = pd.DataFrame(summary_data)
print("\n=== GenePT 1024d Embedding Generation Summary ===")
print(summary_df.to_string(index=False))

# Check for any issues
print(f"\nFiles saved in: {OUTPUT_DIR}")
print(f"Total files: {len(list(OUTPUT_DIR.glob('*.parquet')))}")

# Verify no NaN/Inf values
for tissue_type, embedding_df in tissue_embeddings.items():
    embedding_cols = [col for col in embedding_df.columns if col.startswith('genept_')]
    if embedding_df[embedding_cols].isna().any().any():
        print(f"WARNING: NaN values found in {tissue_type} embeddings")
    if np.isinf(embedding_df[embedding_cols].values).any():
        print(f"WARNING: Inf values found in {tissue_type} embeddings")

print("\nPhase 1 Complete: GenePT 1024d embeddings generated for all tissues!")


=== GenePT 1024d Embedding Generation Summary ===
     Tissue  Cells  Cell_Types  Embedding_Dims  Has_Donor_Info  File_Size_MB
      Blood  17802          19            1024            True         172.5
Bone_Marrow   8045          25            1024            True          76.3
       Lung  11716          31            1024            True         112.3
    Mammary  18539          15            1024            True         179.6
     Thymus   9933          27            1024            True          95.3

Files saved in: /Users/rj/personal/GenePT-tools/data/cz_benchmark/embeddings/genept_1024d
Total files: 5

Phase 1 Complete: GenePT 1024d embeddings generated for all tissues!
